# NagarNetra: train the road-damage detector

This notebook trains a small YOLOv8 model to spot **potholes** and **damaged roads** (cracks) in road photos.
It uses the India part of the public **RDD2022** dataset (7,706 photos, CC BY 4.0).

**Before you start**
1. Open the menu **Runtime > Change runtime type**, choose **T4 GPU**, and press Save.
2. Then run the cells from top to bottom (**Runtime > Run all** works, or press Shift+Enter on each cell).

**How long it takes:** roughly 40 to 60 minutes in total. Most of that is training, and you can leave it alone.
Google Drive is used so that a disconnect does not lose your progress. Colab will ask you to allow Drive access once.

**What you get:** one small file, `road_damage.pt` (about 6 MB). It downloads to your computer at the end and is also
saved in your Google Drive under `nagarnetra/`.

## 1. Check the GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('No GPU found. Use Runtime > Change runtime type > T4 GPU, then run this cell again.')
!nvidia-smi

## 2. Install the tools

In [ ]:
!pip install -q ultralytics remotezip

## 3. Settings
You normally do not need to change these. `EPOCHS` is how many passes over the photos to make.

In [ ]:
WORK = '/content'            # scratch folder for the dataset
EPOCHS = 30                  # training passes (training stops early if it stops improving)
BASE_MODEL = 'yolov8n.pt'    # smallest and fastest YOLOv8, good for a free server
NEGATIVE_SHARE = 0.25        # also keep photos with no damage, as 25% of the damaged photos
EXTRA_TRAIN_ARGS = {}        # left empty; only used for quick tests

## 4. Connect Google Drive (progress is saved here)

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/nagarnetra'
except Exception as error:
    print('Drive was not connected, results will be kept only in this session:', error)
    SAVE_DIR = '/content/saved'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Saving to:', SAVE_DIR)

## 5. Download the India photos
The full dataset is 13 GB, but this cell fetches only the India part (about 0.5 GB). Expect a few minutes.
If it fails, just run the cell again.

In [ ]:
import os, time, zipfile

RAW = f'{WORK}/rdd_raw'
if not os.path.isdir(f'{RAW}/India/train/images'):
    from remotezip import RemoteZip
    URL = 'https://ndownloader.figshare.com/files/38030910'
    os.makedirs(RAW, exist_ok=True)
    for attempt in range(1, 4):
        try:
            with RemoteZip(URL, headers={'User-Agent': 'NagarNetra-hackathon/1.0'}) as z:
                z.extract('RDD2022/India.zip', RAW)
            break
        except Exception as error:
            print('download attempt', attempt, 'failed:', error)
            time.sleep(5)
    else:
        raise RuntimeError('The download failed three times. Run this cell again in a minute.')
    with zipfile.ZipFile(f'{RAW}/RDD2022/India.zip') as zf:
        zf.extractall(RAW)
    os.remove(f'{RAW}/RDD2022/India.zip')

n_images = len(os.listdir(f'{RAW}/India/train/images'))
n_labels = len(os.listdir(f'{RAW}/India/train/annotations/xmls'))
print(f'{n_images} photos, {n_labels} annotation files')

## 6. Prepare the training data
The dataset labels come in RDD's own codes. We keep only what NagarNetra needs and rename it:

| RDD code | Meaning | Our class |
|---|---|---|
| D40 | pothole | `pothole` |
| D00, D10, D20 | cracks (long, cross, alligator) | `damaged_road` |

Other codes (white-line wear, joints, manholes) are dropped. About half of the photos show no damage, so we keep only
a share of them. That is enough to teach the model what a healthy road looks like without drowning out the damage.

In [ ]:
import glob, random, shutil, collections
import xml.etree.ElementTree as ET

CLASS_OF = {'D40': 0, 'D00': 1, 'D10': 1, 'D20': 1}
NAMES = ['pothole', 'damaged_road']
SRC = f'{RAW}/India/train'
OUT = f'{WORK}/road_damage_yolo'


def read_boxes(xml_path):
    root = ET.parse(xml_path).getroot()
    w = float(root.find('size/width').text)
    h = float(root.find('size/height').text)
    lines = []
    for obj in root.findall('object'):
        cls = CLASS_OF.get(obj.find('name').text)
        if cls is None:
            continue
        box = obj.find('bndbox')
        xa, ya, xb, yb = (float(box.find(k).text) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
        x1, x2 = max(0.0, min(xa, xb)), min(w, max(xa, xb))
        y1, y2 = max(0.0, min(ya, yb)), min(h, max(ya, yb))
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        lines.append(f'{cls} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}')
    return lines


random.seed(0)
stems = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{SRC}/annotations/xmls/*.xml'))
boxes = {s: read_boxes(f'{SRC}/annotations/xmls/{s}.xml') for s in stems}
damaged = [s for s in stems if boxes[s]]
healthy = [s for s in stems if not boxes[s]]
random.shuffle(healthy)
chosen = damaged + healthy[: int(len(damaged) * NEGATIVE_SHARE)]
random.shuffle(chosen)
n_val = int(len(chosen) * 0.1)
splits = {'val': chosen[:n_val], 'train': chosen[n_val:]}

shutil.rmtree(OUT, ignore_errors=True)
for split, split_stems in splits.items():
    os.makedirs(f'{OUT}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUT}/labels/{split}', exist_ok=True)
    for s in split_stems:
        shutil.copy(f'{SRC}/images/{s}.jpg', f'{OUT}/images/{split}/{s}.jpg')
        with open(f'{OUT}/labels/{split}/{s}.txt', 'w') as f:
            f.write('\n'.join(boxes[s]))

with open(f'{OUT}/data.yaml', 'w') as f:
    f.write(f'path: {OUT}\ntrain: images/train\nval: images/val\nnames:\n  0: pothole\n  1: damaged_road\n')

print(f'{len(damaged)} photos with damage, {len(healthy)} without (kept {len(chosen) - len(damaged)} of those)')
for split, split_stems in splits.items():
    counts = collections.Counter(int(line.split()[0]) for s in split_stems for line in boxes[s])
    print(f'{split:5} {len(split_stems):5} photos   ' + '   '.join(f'{NAMES[c]}: {counts[c]}' for c in range(2)))

## 7. Train
This is the long step. A line of numbers is printed for each pass. Watch **mAP50**: higher is better, and it should
climb over the first 10 to 15 passes. Training stops by itself if it stops improving.

If the connection drops, reconnect, run cells 3 to 6 again, then run the **resume** cell below instead of this one.

In [ ]:
from ultralytics import YOLO

RUNS = f'{SAVE_DIR}/runs'
train_args = dict(
    data=f'{OUT}/data.yaml', epochs=EPOCHS, imgsz=640, batch=32, patience=8,
    project=RUNS, name='road_damage', exist_ok=True, workers=2, seed=0, plots=True,
)
train_args.update(EXTRA_TRAIN_ARGS)

model = YOLO(BASE_MODEL)
model.train(**train_args)

In [ ]:
# Only run this if training was interrupted. It continues from the last saved pass.
# from ultralytics import YOLO
# YOLO(f'{SAVE_DIR}/runs/road_damage/weights/last.pt').train(resume=True)

## 8. How good is it?

In [ ]:
best = f'{RUNS}/road_damage/weights/best.pt'
metrics = YOLO(best).val(data=f'{OUT}/data.yaml', imgsz=640, plots=False, verbose=False)
print(f'{"class":14}{"precision":>10}{"recall":>10}{"mAP50":>10}')
for i, name in enumerate(NAMES):
    p, r, ap50, _ = metrics.box.class_result(i)
    print(f'{name:14}{p:10.2f}{r:10.2f}{ap50:10.2f}')
print(f'overall mAP50: {metrics.box.map50:.2f}   (mAP50-95: {metrics.box.map:.2f})')

**Reading the numbers:** *precision* is how often an alert is right, *recall* is how much real damage it finds,
and *mAP50* is a single score combining both (1.00 is perfect). In our own run (30 passes on an RTX 3050) the scores were mAP50 0.38 for `pothole` and 0.52 for `damaged_road`,
0.45 overall. Treat that as a normal first version, not a finished one.

## 9. See it work on photos it has not trained on

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

val_images = sorted(glob.glob(f'{OUT}/images/val/*.jpg'))
with_damage = [p for p in val_images if os.path.getsize(p.replace('/images/', '/labels/').replace('.jpg', '.txt')) > 0]
sample = random.sample(with_damage, min(6, len(with_damage)))
predictions = YOLO(best).predict(sample, conf=0.25, imgsz=640, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, result in zip(axes.flat, predictions):
    ax.imshow(result.plot()[..., ::-1])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 10. Save the model

In [ ]:
FINAL = f'{WORK}/road_damage.pt'
shutil.copy(best, FINAL)
shutil.copy(best, f'{SAVE_DIR}/road_damage.pt')
print(f'Saved road_damage.pt ({os.path.getsize(FINAL) / 1e6:.1f} MB) to {SAVE_DIR}')

In [ ]:
from google.colab import files
files.download(FINAL)

## Done
Your browser should have downloaded `road_damage.pt`. A copy is also in Google Drive under `nagarnetra/`.

**Next step:** put the file in `nagarnetra/backend/models/` and tell Claude it is there. The backend will use it to flag
potholes and damaged roads on every camera frame.